<!-- Architecture
Image Input → LangChain → [Gemini / HuggingFace / OpenRouter / OpenAI] → Text Output -->

In [ ]:
# Image to Text - LangChain + HuggingFace Inference Providers
# Uses ChatOpenAI pointed at HF's OpenAI-compatible router (the only reliable LangChain path)
# Models: aya-vision-32b (cohere)

import os
import base64
from dotenv import load_dotenv
from pydantic import SecretStr
from langchain_openai import ChatOpenAI          # pip install langchain-openai
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

# ─────────────────────────────────────────────
# Confirmed FREE vision models for HF Pro users
# Format for HF router: "model_id:provider"
# ─────────────────────────────────────────────
MODELS = {
    "aya": {
        "model":    "CohereLabs/aya-vision-32b",
        "provider": "cohere",
        "router_model": "CohereLabs/aya-vision-32b:cohere",
    },
}

TEST_IMAGE_PATH = "C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Assignment\Access_Different_LLMs\images\AIImage.jpg"


# ─────────────────────────────────────────────
# Helper: local path → base64 data URI
# ─────────────────────────────────────────────
def load_image(image_source: str) -> str:
    """Convert local file to base64 data URI. or pass image as URL directly."""

    if not os.path.exists(image_source):
        raise FileNotFoundError(f"Image not found: {image_source}")

    with open(image_source, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    ext = image_source.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
            "png": "image/png",  "gif": "image/gif",
            "webp": "image/webp"}.get(ext, "image/jpeg")

    return f"data:{mime};base64,{encoded}"


# ─────────────────────────────────────────────
# Chain builder — uses HF's OpenAI-compatible router
# LangChain's HuggingFaceEndpoint does NOT support
# the new Inference Providers system, so we use
# ChatOpenAI pointed at https://router.huggingface.co/v1
# ─────────────────────────────────────────────
def create_vision_chain(router_model: str):
    llm = ChatOpenAI(
        model=router_model,
        base_url="https://router.huggingface.co/v1",    # HF's OpenAI-compatible router
        api_key=SecretStr(HF_TOKEN),
        max_tokens=512,
    )
    return llm | StrOutputParser()


# ─────────────────────────────────────────────
# Main inference function
# ─────────────────────────────────────────────
def generate_text_from_image(
    image_source: str,
    prompt: str = "Describe this image in detail.",
    model_key: str = "aya",          # "qwen" or "aya"
) -> str:
    config = MODELS[model_key]
    print(f"\n{'='*50}")
    print(f"Model    : {config['model']}")
    print(f"Provider : {config['provider']}")
    print(f"Image    : {image_source}")
    print(f"Prompt   : {prompt}")
    print(f"{'='*50}\n")

    image_url = load_image(image_source)

    message = HumanMessage(
        content=[
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": prompt},
        ]
    )

    chain = create_vision_chain(config["router_model"])
    return chain.invoke([message])


# ─────────────────────────────────────────────
# Run both models and compare outputs
# ─────────────────────────────────────────────
if __name__ == "__main__":
    PROMPT = "Describe this image in detail."

    # ── Model 2: aya-vision-32b (cohere) ──
    print(">>> Running Model 2: aya-vision-32b (cohere)")
    try:
        output2 = generate_text_from_image(
            image_source=TEST_IMAGE_PATH,
            prompt=PROMPT,
            model_key="aya",
        )
        print("--- Aya Vision Output ---")
        print(output2)
    except Exception as e:
        print(f"[Aya Error] {e}")

>>> Running Model 2: aya-vision-32b (cohere)

Model    : CohereLabs/aya-vision-32b
Provider : cohere
Image    : C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Assignment\Access_Different_LLMs\images\AIImage.jpg
Prompt   : Describe this image in detail.

--- Aya Vision Output ---
This image is a vibrant and complex digital illustration that captures the essence of artificial intelligence (AI) and its integration into various aspects of modern life. At the center of the composition is a stylized representation of a human brain, which serves as a metaphor for AI's cognitive abilities. The brain is depicted in a gradient of blue hues, with intricate neural pathways and synapses highlighted in white, giving it a futuristic and technological appearance.

Surrounding the brain are numerous icons and symbols that represent different industries and applications where AI is making an impact. These include:

1. **Technology**: Icons of a laptop, smartphone, and a circuit board sign